# simscDNAm

正式主模型：条件VAE及潜空间扩散生成 → 同Subclass内部fit真实缺失掩码 → 联合秩投影和经验分位数映射。固定因子上限32、投影强度1，所有区域应用相同规则，保留NaN位置。训练和映射均只使用内部fit真实值。

在simscDNAm环境的Jupyter中从上到下运行；四方法统一比较使用`evaluate/A1、A2、A3/evaluate.ipynb`。`PRETRAINED_RUNS`为空时训练新模型，指定既有训练目录时复用CVAE＋mask并重新拟合正式对齐。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import gc, hashlib, json, math, random, sys, warnings, uuid, shutil
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
sys.path.insert(0, str(PROJECT))
import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader
from model.src.config import TrainingConfig
from data.training import (FullMethylationDataset, load_training_input, load_target_labels,
                       internal_subclass_split, fit_feature_medians, validate_real_duplicates)
from model.src.model import LatentDiffusionVAE, latent_diffusion_vae_loss, sample_latent_ddim, load_cvae_model
from data.training import _close_backed, _dense_float32
import os
from evaluate.A3.metrics import milisi

from scipy.stats import rankdata
from scipy.special import ndtri
from threadpoolctl import threadpool_limits
from IPython.display import display


In [ ]:
def write_synthetic_h5ad(
    output_path: Path,
    model: LatentDiffusionVAE,
    synthetic_class_indices: np.ndarray,
    synthetic_subclass: np.ndarray,
    var: pd.DataFrame,
    feature_names: np.ndarray,
    sampling_steps: int,
    device: torch.device,
    seed: int,
    row_block: int = 8,
    feature_block: int = 4096,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    n_samples = len(synthetic_subclass)
    obs = pd.DataFrame(
        {"origin": "Synthetic", "generator": "End-to-end conditional latent diffusion VAE", "Subclass": synthetic_subclass},
        index=pd.Index([f"synthetic_{i:06d}" for i in range(1, n_samples + 1)], name="cell_id"),
    )
    partial = output_path.with_name(
        f".{output_path.name}.{os.getpid()}.{uuid.uuid4().hex}.partial"
    )
    shell = ad.AnnData(
        X=sp.csr_matrix((n_samples, len(feature_names)), dtype=np.float32),
        obs=obs,
        var=var.copy(),
    )
    shell.var_names = pd.Index(feature_names.astype(str))
    shell.write_h5ad(partial, compression="gzip")
    model.eval()
    with h5py.File(partial, "r+", locking=False) as handle:
        del handle["X"]
        dataset = handle.create_dataset(
            "X", shape=(n_samples, len(feature_names)), dtype="float32",
            chunks=(min(row_block, n_samples), min(feature_block, len(feature_names))),
            compression="gzip", compression_opts=4, shuffle=True,
        )
        dataset.attrs["encoding-type"] = "array"
        dataset.attrs["encoding-version"] = "0.2.0"
        with torch.no_grad():
            for start in range(0, n_samples, row_block):
                stop = min(start + row_block, n_samples)
                condition = torch.from_numpy(
                    synthetic_class_indices[start:stop]
                ).long().to(device)
                latent = sample_latent_ddim(
                    model, condition, sampling_steps, seed + start
                )
                values = model.decode(latent).cpu().numpy().astype(np.float32)
                dataset[start:stop] = values
    partial.replace(output_path)


def validate_synthetic_h5ad(
    path: Path, feature_names: np.ndarray, expected_count: int
) -> dict[str, float | int]:
    item = ad.read_h5ad(path, backed="r")
    try:
        if item.shape != (expected_count, len(feature_names)):
            raise AssertionError(f"合成数据形状不符合预期： {item.shape}")
        if not np.array_equal(item.var_names.astype(str).to_numpy(), feature_names.astype(str)):
            raise AssertionError("合成数据与输入特征轴不一致")
        minimum, maximum = math.inf, -math.inf
        for start in range(0, item.n_obs, 32):
            block = _dense_float32(item.X[start:min(start + 32, item.n_obs)])
            if not np.isfinite(block).all():
                raise AssertionError("合成数据包含非有限值")
            minimum = min(minimum, float(block.min()))
            maximum = max(maximum, float(block.max()))
        if minimum < 0 or maximum > 1:
            raise AssertionError("合成值超出 [0, 1]")
        return {
            "n_cells": int(item.n_obs), "n_features": int(item.n_vars),
            "minimum": minimum, "maximum": maximum,
        }
    finally:
        _close_backed(item)

In [ ]:
def fit_selection_reference(matrix, fit_rows, validation_rows, labels,
                            natural_count, top_regions=50000, components=50):
    """只在内部训练细胞的原始区域拟合参照；验证矩阵只做投影。"""
    fit_rows, validation_rows = np.asarray(fit_rows), np.asarray(validation_rows)
    if np.intersect1d(fit_rows, validation_rows).size:
        raise ValueError("内部训练与验证细胞重叠")
    fit_labels = np.asarray(labels)[fit_rows]
    levels = np.unique(fit_labels)
    if len(levels) < 2 or len(levels) >= len(fit_rows):
        raise ValueError("内部训练的 ANOVA 需要至少两类且有重复细胞")
    fill = np.full(natural_count, np.nan)
    scores = np.full(natural_count, np.nan)
    for start in range(0, natural_count, 512):
        columns = np.arange(start, min(start + 512, natural_count))
        values = np.array(matrix[np.ix_(fit_rows, columns)], dtype=np.float64)
        available = np.isfinite(values).any(0)
        medians = np.full(len(columns), np.nan)
        medians[available] = np.nanmedian(values[:, available], axis=0)
        fill[columns] = medians
        values = values[:, available]
        np.copyto(values, medians[available][None, :], where=~np.isfinite(values))
        grand = values.mean(0)
        between, within = np.zeros(values.shape[1]), np.zeros(values.shape[1])
        for level in levels:
            group = values[fit_labels == level]
            mean = group.mean(0)
            between += len(group) * (mean - grand) ** 2
            within += ((group - mean) ** 2).sum(0)
        numerator, denominator = between / (len(levels) - 1), within / (len(fit_rows) - len(levels))
        local = np.divide(numerator, denominator, out=np.full(values.shape[1], np.nan), where=denominator > 0)
        local[(denominator == 0) & (numerator > 0)] = np.inf
        scores[columns[available]] = local
    eligible = np.flatnonzero(np.isfinite(scores) | np.isposinf(scores))
    if len(eligible) < 2:
        raise ValueError("内部训练没有足够的可评分原始区域")
    selected = np.sort(eligible[np.argsort(scores[eligible], kind="stable")[-min(top_regions, len(eligible)):]])
    fill[~np.isfinite(fill)] = np.median(fill[np.isfinite(fill)])
    def prepared(rows):
        values = np.array(matrix[np.ix_(rows, selected)], dtype=np.float32)
        np.copyto(values, fill[selected][None, :], where=~np.isfinite(values))
        return values
    pca = PCA(n_components=min(components, len(fit_rows)-1, len(selected)),
              svd_solver="randomized", random_state=42)
    pca.fit(prepared(fit_rows))
    return {"selected": selected, "fill": fill, "pca": pca,
            "validation_rep": pca.transform(prepared(validation_rows)),
            "validation_labels": np.asarray(labels)[validation_rows],
            "fit_cells": len(fit_rows), "validation_cells": len(validation_rows)}


def selection_rank(record):
    """通过原验收的候选优先；只用内部验证 miLISI 排名。"""
    score = float(record["selection_milisi"])
    return (not bool(record["selection_eligible"]) or not np.isfinite(score),
            -score if np.isfinite(score) else float("inf"), float(record["selection_score"]))


def guard_run_identity(directory, identity):
    """签名在读取/覆盖缓存之前检查；不同输入、代码或配置不允许恢复。"""
    directory = Path(directory)
    path = directory / "run_identity.json"
    normalized = json.loads(json.dumps(identity, sort_keys=True, ensure_ascii=False))
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != normalized:
        raise ValueError("运行签名变化，请使用新输出目录；禁止复用旧模型或合成矩阵")
    if not path.exists() and any(directory.glob("*/diagnostics.json")):
        raise ValueError("已有候选缺少运行签名，请使用新输出目录")
    directory.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        write_json(path, normalized)


def vae_state_hash(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        if not name.startswith("denoiser."):
            digest.update(name.encode())
            digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def source_neighborhoods(real_rep, generated_rep, k=30):
    """与正式指标同一邻域，额外报告两侧混合，防止单侧混合被汇总掩盖。"""
    combined = np.vstack([real_rep, generated_rep])
    raw = NearestNeighbors(n_neighbors=k+1).fit(combined).kneighbors(combined, return_distance=False)
    indices = np.array([row[row != i][:k] for i, row in enumerate(raw)])
    p_real = (indices < len(real_rep)).mean(1)
    return {"真实侧异来源近邻比例": float((1-p_real[:len(real_rep)]).mean()),
            "合成侧异来源近邻比例": float(p_real[len(real_rep):].mean()),
            "合成侧全同源邻域比例": float((p_real[len(real_rep):] == 0).mean())}


In [ ]:
def train_candidate(matrix, retained_rows, medians, labels, class_names,
                    train_positions, validation_positions, beta_target,
                    candidate_dir, device, checkpoint_metadata, config, *, selection_reference, frozen_vae=True):
    # 以局部配置绑定原 notebook 常量；不同运行之间不共享可变全局配置。
    DEVICE = device
    SEED = config.SEED
    LATENT_DIM = config.LATENT_DIM
    CONDITION_DIM = config.CONDITION_DIM
    HIDDEN_DIM = config.HIDDEN_DIM
    DENOISER_BLOCKS = config.DENOISER_BLOCKS
    DIFFUSION_TIMESTEPS = config.DIFFUSION_TIMESTEPS
    DDIM_SAMPLING_STEPS = config.DDIM_SAMPLING_STEPS
    DIFFUSION_LOSS_WEIGHT = config.DIFFUSION_LOSS_WEIGHT
    DIFFUSION_PREDICTION = config.DIFFUSION_PREDICTION
    DIFFUSION_SCHEDULE = config.DIFFUSION_SCHEDULE
    DROPOUT_RATE = config.DROPOUT_RATE
    FEATURE_BLOCK = config.FEATURE_BLOCK
    TRAIN_BATCH_SIZE = config.TRAIN_BATCH_SIZE
    VAE_PRETRAIN_EPOCHS = config.VAE_PRETRAIN_EPOCHS
    DIFFUSION_WARMUP_EPOCHS = config.DIFFUSION_WARMUP_EPOCHS
    JOINT_EPOCHS = config.JOINT_EPOCHS
    JOINT_VAE_LR_FACTOR = config.JOINT_VAE_LR_FACTOR
    SELECTION_INTERVAL = config.SELECTION_INTERVAL
    SELECTION_FEATURES = config.SELECTION_FEATURES
    DIFFUSION_PATIENCE = config.DIFFUSION_PATIENCE
    PATIENCE = config.PATIENCE
    EARLY_STOP_MIN_DELTA = config.EARLY_STOP_MIN_DELTA
    LEARNING_RATE = config.LEARNING_RATE
    KL_WARMUP_EPOCHS = config.KL_WARMUP_EPOCHS

    def _seed_everything(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


    def _make_loaders(
        matrix: np.memmap,
        retained_rows: np.ndarray,
        medians: np.ndarray,
        labels: np.ndarray,
        train_positions: np.ndarray,
        validation_positions: np.ndarray,
    ) -> tuple[DataLoader, DataLoader]:
        generator = torch.Generator().manual_seed(SEED)
        train_dataset = FullMethylationDataset(
            matrix, retained_rows, train_positions, medians, labels
        )
        validation_dataset = FullMethylationDataset(
            matrix, retained_rows, validation_positions, medians, labels
        )
        common = {"batch_size": TRAIN_BATCH_SIZE, "num_workers": 0, "pin_memory": DEVICE.type == "cuda"}
        return (
            DataLoader(train_dataset, shuffle=True, generator=generator, **common),
            DataLoader(validation_dataset, shuffle=False, **common),
        )


    def _run_epoch(
        model, loader, optimizer, beta, device, random_seed=None,
        stage="joint", diffusion_weight=DIFFUSION_LOSS_WEIGHT,
    ):
        training = optimizer is not None
        model.train(training)
        sums = np.zeros(4, dtype=np.float64)
        observed_count = cell_count = 0
        generator = None if random_seed is None else torch.Generator(device=device).manual_seed(random_seed)
        with torch.set_grad_enabled(training):
            for batch, observed, classes in loader:
                batch, observed, classes = [value.to(device) for value in (batch, observed, classes)]
                if training:
                    optimizer.zero_grad(set_to_none=True)
                mu, logvar = model.encode(batch)
                noise = torch.randn(mu.shape, device=device, generator=generator)
                z = model.reparameterize(mu, logvar, noise)
                reconstruction = model.decode(z)
                z0 = model.normalize_latent(z)
                if stage == "vae_pretrain":
                    prediction = target = torch.zeros_like(z0)
                else:
                    times = torch.randint(model.diffusion_timesteps, (len(batch),), device=device, generator=generator)
                    epsilon = torch.randn(z0.shape, device=device, generator=generator)
                    prediction = model.predict_velocity(model.add_noise(z0, times, epsilon), times, classes)
                    target = model.velocity_target(z0, times, epsilon)
                loss, recon, kl, diffusion = latent_diffusion_vae_loss(
                    reconstruction, batch, observed, mu, logvar, prediction, target,
                    beta, 0.0 if stage == "vae_pretrain" else diffusion_weight,
                )
                if not torch.isfinite(loss):
                    raise FloatingPointError("训练损失出现非有限值")
                if training:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    optimizer.step()
                count = int(observed.sum())
                sums += np.array([recon.item() * count, kl.item() * len(batch), diffusion.item() * len(batch), 0.0])
                observed_count += count
                cell_count += len(batch)
        recon, kl, diffusion = sums[:3] / np.array([observed_count, cell_count, cell_count])
        weight = 0.0 if stage == "vae_pretrain" else diffusion_weight
        return {"loss": float(recon + beta * kl + weight * diffusion), "reconstruction": float(recon), "kl": float(kl), "diffusion": float(diffusion)}


    def fit_model_statistics(matrix, rows, medians):
        # 只读取内部训练子集；验证细胞不参与填补、中心化或评估特征选择。
        center = np.empty(matrix.shape[1], dtype=np.float32)
        variance = np.empty_like(center)
        for start in range(0, matrix.shape[1], FEATURE_BLOCK):
            stop = min(start + FEATURE_BLOCK, matrix.shape[1])
            values = np.asarray(matrix[np.ix_(rows, np.arange(start, stop))], dtype=np.float32)
            mask = np.isfinite(values)
            count = mask.sum(0)
            values = np.where(mask, values, 0.0)
            mean = values.sum(0, dtype=np.float64) / np.maximum(count, 1)
            center[start:stop] = np.where(count, mean, medians[start:stop])
            variance[start:stop] = np.maximum(np.square(values, dtype=np.float64).sum(0) / np.maximum(count, 1) - mean ** 2, 0)
        features = np.sort(np.argsort(variance)[-min(SELECTION_FEATURES, len(variance)):])
        return center, features


    @torch.no_grad()
    def cache_posterior(model, loader, device):
        model.eval()
        means, logvars, labels = [], [], []
        for batch, _, classes in loader:
            mu, logvar = model.encode(batch.to(device))
            means.append(mu.detach())
            logvars.append(logvar.detach())
            labels.append(classes.to(device))
        return torch.cat(means), torch.cat(logvars), torch.cat(labels)


    def diffusion_cached_epoch(model, cached, optimizer, device, seed=None):
        training = optimizer is not None
        model.eval()
        model.denoiser.train(training)
        mu, logvar, labels = cached
        generator = None if seed is None else torch.Generator(device=device).manual_seed(seed)
        order = torch.randperm(len(mu), device=device, generator=generator) if training else torch.arange(len(mu), device=device)
        total = 0.0
        with torch.set_grad_enabled(training):
            for indices in order.split(TRAIN_BATCH_SIZE):
                posterior_noise = torch.randn(mu[indices].shape, device=device, generator=generator)
                z0 = model.normalize_latent(model.reparameterize(mu[indices], logvar[indices], posterior_noise).detach())
                times = torch.randint(model.diffusion_timesteps, (len(indices),), device=device, generator=generator)
                epsilon = torch.randn(z0.shape, device=device, generator=generator)
                prediction = model.predict_velocity(model.add_noise(z0, times, epsilon), times, labels[indices])
                loss = (prediction - model.velocity_target(z0, times, epsilon)).square().mean()
                if not torch.isfinite(loss):
                    raise FloatingPointError("扩散预热损失出现非有限值")
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.denoiser.parameters(), 5.0)
                    optimizer.step()
                total += loss.item() * len(indices)
        return total / len(mu)


    @torch.no_grad()
    def evaluate_representation(model, loader, device):
        model.eval()
        errors = np.zeros(3, dtype=np.float64)
        count = 0
        means = []
        for batch, observed, _ in loader:
            batch, observed = batch.to(device), observed.to(device)
            mu, _ = model.encode(batch)
            for index, output in enumerate((model.decode(mu), model.decode(mu.roll(1, 0)), model.input_center.expand_as(batch))):
                errors[index] += (output - batch).square().masked_select(observed).sum().item()
            count += int(observed.sum())
            means.append(mu.cpu())
        recon, shuffled, baseline = errors / count
        latent_variance = torch.cat(means).var(dim=0, unbiased=False)
        return {
            "mean_reconstruction": float(recon), "shuffle_reconstruction": float(shuffled),
            "baseline_reconstruction": float(baseline),
            "reconstruction_gain": float(1 - recon / max(baseline, 1e-12)),
            "shuffle_gain": float((shuffled - recon) / max(baseline, 1e-12)),
            "active_units": int((latent_variance > 1e-3).sum()),
        }


    @torch.no_grad()
    def evaluate_selection_score(model, loader, device, features):
        model.eval()
        validation = _run_epoch(model, loader, None, 0.0, device, SEED + 20_000)
        representation = evaluate_representation(model, loader, device)
        real, observed, labels = [], [], []
        for batch, mask, classes in loader:
            real.append(batch[:, features].numpy())
            observed.append(mask[:, features].numpy())
            labels.append(classes.numpy())
        real, observed, labels = np.concatenate(real), np.concatenate(observed), np.concatenate(labels)

        def generate(classes):
            pieces = []
            for start in range(0, len(classes), 64):
                condition = torch.as_tensor(classes[start:start + 64], device=device)
                latent = sample_latent_ddim(model, condition, DDIM_SAMPLING_STEPS, SEED + 30_000 + start)
                pieces.append(model.decode(latent)[:, output_features].cpu().numpy())
            return np.concatenate(pieces)

        scoring_features = np.asarray(selection_reference["selected"])
        output_features = np.union1d(features, scoring_features)
        old_positions = np.searchsorted(output_features, features)
        scoring_positions = np.searchsorted(output_features, scoring_features)
        # 生成同一批完整解码结果中的所需列，不改网络、不改采样器。
        generated_all = generate(labels)
        generated = generated_all[:, old_positions]
        generated_rep = selection_reference["pca"].transform(generated_all[:, scoring_positions])
        validation_milisi, _, _ = milisi(selection_reference["validation_rep"], labels,
                                         generated_rep, labels, n_neighbors=30)
        permuted_labels = np.random.default_rng(SEED).permutation(labels)
        permuted = generate(permuted_labels)[:, old_positions]
        counts = observed.sum(0)
        mean = (real * observed).sum(0) / np.maximum(counts, 1)
        variance = (((real - mean) ** 2) * observed).sum(0) / np.maximum(counts, 1)
        valid = (counts >= 10) & (variance > 1e-4)
        if not valid.any():
            raise ValueError("验证集中没有足够的可观测变异特征")
        scale = np.maximum(variance[valid], 1e-4)
        generated_variance = generated[:, valid].var(0)
        mean_error = float(np.mean((generated[:, valid].mean(0) - mean[valid]) ** 2 / scale))
        variance_error = float(np.mean(np.abs(np.log((generated_variance + 1e-6) / (scale + 1e-6)))))
        diversity_ratio = float(np.median(np.sqrt(generated_variance / scale)))

        def centroid_error(samples):
            errors = []
            for label in np.unique(labels):
                rows = labels == label
                if rows.sum() < 3:
                    continue
                mask = observed[rows][:, valid]
                count = mask.sum(0)
                keep = count >= 2
                if not keep.any():
                    continue
                reference = (real[rows][:, valid] * mask).sum(0) / np.maximum(count, 1)
                errors.append(np.mean(((samples[rows][:, valid].mean(0) - reference) ** 2 / scale)[keep]))
            return float(np.mean(errors)) if errors else 0.0

        conditional_error = centroid_error(generated)
        eligible = bool(representation["reconstruction_gain"] > 0.005 and representation["shuffle_gain"] > 0.005 and representation["active_units"] >= 4 and diversity_ratio > 0.1)
        return {
            **representation,
            "selection_milisi": float(validation_milisi),
            **source_neighborhoods(selection_reference["validation_rep"], generated_rep),
            "selection_reconstruction": validation["reconstruction"],
            "selection_kl": validation["kl"], "selection_diffusion": validation["diffusion"],
            "selection_mean_error": mean_error, "selection_variance_error": variance_error,
            "selection_centroid_error": conditional_error, "selection_diversity_ratio": diversity_ratio,
            "condition_permuted_centroid_error": centroid_error(permuted),
            "condition_output_rmse": float(np.sqrt(np.mean((generated - permuted) ** 2))),
            "selection_eligible": eligible,
            "selection_score": float(validation["reconstruction"] / max(representation["baseline_reconstruction"], 1e-12) + mean_error + variance_error + conditional_error),
        }


    def train_cvae_candidate(matrix, retained_rows, medians, labels, class_names,
                             train_positions, validation_positions, beta_target,
                             candidate_dir, device, checkpoint_metadata):
        _seed_everything(SEED)
        candidate_dir.mkdir(parents=True, exist_ok=True)
        train_loader, validation_loader = _make_loaders(matrix, retained_rows, medians, labels, train_positions, validation_positions)
        model = LatentDiffusionVAE(matrix.shape[1], len(class_names), LATENT_DIM, CONDITION_DIM,
                                  HIDDEN_DIM, DROPOUT_RATE, DIFFUSION_TIMESTEPS, DENOISER_BLOCKS).to(device)
        center, features = fit_model_statistics(matrix, retained_rows[train_positions], medians)
        model.input_center.copy_(torch.as_tensor(center, device=device))
        # 用训练集均值初始化输出截距；细胞残差仍必须由潜变量学习。
        with torch.no_grad():
            model.decoder[-2].bias.copy_(torch.logit(model.input_center.clamp(1e-4, 1 - 1e-4)))
        vae_parameters = [p for name, p in model.named_parameters() if not name.startswith("denoiser.")]
        history = []
        selection_history = []
        history_path = candidate_dir / "training_history.csv"
        best_epoch = 0

        def snapshot():
            return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        def log(stage, local_epoch, train, validation, extra=None):
            record = {"epoch": len(history) + 1, "stage": stage, "stage_epoch": local_epoch,
                      "beta_target": beta_target, **{f"train_{k}": v for k, v in train.items()},
                      **{f"validation_{k}": v for k, v in validation.items()}, **(extra or {})}
            history.append(record)
            pd.DataFrame(history).to_csv(history_path, index=False)
            print(f"β={beta_target:g} 阶段={stage} 轮次={local_epoch} 验证={validation.get('loss', float('nan')):.6f} "
                  f"重构={validation.get('reconstruction', float('nan')):.6f} 扩散={validation.get('diffusion', float('nan')):.6f}", flush=True)

        optimizer = torch.optim.Adam(vae_parameters, lr=LEARNING_RATE)
        best_loss, stale, best_state = math.inf, 0, snapshot()
        for epoch in range(1, VAE_PRETRAIN_EPOCHS + 1):
            beta = beta_target * min(1.0, max(0, epoch - 5) / KL_WARMUP_EPOCHS)
            train = _run_epoch(model, train_loader, optimizer, beta, device, stage="vae_pretrain")
            val = _run_epoch(model, validation_loader, None, beta, device, SEED + 10_000, stage="vae_pretrain")
            log("vae_pretrain", epoch, train, val)
            if epoch < 5 + KL_WARMUP_EPOCHS:
                continue
            if val["reconstruction"] < best_loss - EARLY_STOP_MIN_DELTA:
                best_loss, stale, best_state = val["reconstruction"], 0, snapshot()
            else:
                stale += 1
            if stale >= PATIENCE:
                break
        model.load_state_dict(best_state)
        del optimizer, best_state
        representation = evaluate_representation(model, validation_loader, device)
        print("VAE 预训练检查", json.dumps(representation, ensure_ascii=False), flush=True)
        (candidate_dir / "pretrain_diagnostics.json").write_text(json.dumps(representation, indent=2), encoding="utf-8")

        cached_train = cache_posterior(model, train_loader, device)
        cached_validation = cache_posterior(model, validation_loader, device)
        with torch.no_grad():
            mu, logvar, _ = cached_train
            model.latent_center.copy_(mu.mean(0))
            model.latent_scale.copy_((mu.var(0, unbiased=False) + logvar.exp().mean(0)).sqrt().clamp_min(1e-3))
        for parameter in vae_parameters:
            parameter.requires_grad_(False)
        vae_before = vae_state_hash(model)
        optimizer = torch.optim.Adam(model.denoiser.parameters(), lr=LEARNING_RATE)
        best_denoiser, best_selection, best_rank = None, None, (True, math.inf, math.inf)
        last_improvement = 0
        best_diffusion_epoch = 0
        for epoch in range(1, DIFFUSION_WARMUP_EPOCHS + 1):
            train = diffusion_cached_epoch(model, cached_train, optimizer, device)
            val = diffusion_cached_epoch(model, cached_validation, None, device, SEED + 10_000)
            if epoch == 1 or epoch % 10 == 0:
                log("diffusion_warmup", epoch, {"loss": train, "diffusion": train}, {"loss": val, "diffusion": val})
            if epoch == 1 or epoch % SELECTION_INTERVAL == 0 or epoch == DIFFUSION_WARMUP_EPOCHS:
                selection = evaluate_selection_score(model, validation_loader, device, features)
                selection_history.append({"stage": "diffusion_warmup", "stage_epoch": epoch, **selection})
                pd.DataFrame(selection_history).to_csv(candidate_dir / "selection_history.csv", index=False)
                rank = selection_rank(selection)
                print(f"扩散轮次 {epoch}；内部验证 miLISI={selection['selection_milisi']:.6f}；通过验收={selection['selection_eligible']}", flush=True)
                if best_denoiser is None or rank < best_rank:
                    best_rank, best_selection, last_improvement = rank, selection, epoch
                    best_diffusion_epoch = epoch
                    best_denoiser = {k: v.detach().cpu().clone() for k, v in model.denoiser.state_dict().items()}
            if epoch >= 100 and epoch - last_improvement >= DIFFUSION_PATIENCE:
                break
        diffusion_epochs_completed = epoch
        model.denoiser.load_state_dict(best_denoiser)
        del cached_train, cached_validation, optimizer, best_denoiser
        if frozen_vae:
            best_state = snapshot()
            best_epoch = best_diffusion_epoch
        else:
            for parameter in vae_parameters:
                parameter.requires_grad_(True)

            optimizer = torch.optim.Adam([
                {"params": vae_parameters, "lr": LEARNING_RATE * JOINT_VAE_LR_FACTOR},
                {"params": model.denoiser.parameters(), "lr": LEARNING_RATE},
            ])
            best_state, best_selection, stale = None, None, 0
            for epoch in range(1, JOINT_EPOCHS + 1):
                weight = DIFFUSION_LOSS_WEIGHT * min(1.0, epoch / 10)
                train = _run_epoch(model, train_loader, optimizer, beta_target, device, diffusion_weight=weight)
                val = _run_epoch(model, validation_loader, None, beta_target, device, SEED + 10_000, diffusion_weight=weight)
                log("joint", epoch, train, val, {"diffusion_weight": weight})
                if epoch % SELECTION_INTERVAL and epoch != JOINT_EPOCHS:
                    continue
                selection = evaluate_selection_score(model, validation_loader, device, features)
                selection_history.append({"stage": "joint", "stage_epoch": epoch, **selection})
                pd.DataFrame(selection_history).to_csv(candidate_dir / "selection_history.csv", index=False)
                print("内部验证诊断", json.dumps(selection, ensure_ascii=False), flush=True)
                # 先比较是否通过塌缩检查，再比较生成质量，绝不按原始 KL 排名。
                rank = selection_rank(selection)
                old_rank = selection_rank(best_selection) if best_selection else (True, math.inf, math.inf)
                if best_state is None or rank < old_rank:
                    best_state, best_selection, stale = snapshot(), selection, 0
                    best_epoch = len(history)
                else:
                    stale += 1
                if epoch >= 20 and stale >= PATIENCE:
                    break
        model.load_state_dict(best_state)
        selection = evaluate_selection_score(model, validation_loader, device, features)
        vae_after = vae_state_hash(model)
        if frozen_vae and vae_before != vae_after:
            raise AssertionError("冻结扩散训练修改了 VAE 参数或统计量")
        model_path = candidate_dir / "model.pt"
        checkpoint = {
            "state_dict": best_state, "input_dim": int(matrix.shape[1]), "latent_dim": LATENT_DIM,
            "condition_dim": CONDITION_DIM, "hidden_dim": HIDDEN_DIM, "dropout_rate": DROPOUT_RATE,
            "denoiser_blocks": DENOISER_BLOCKS, "diffusion_timesteps": DIFFUSION_TIMESTEPS,
            "sampling_steps": DDIM_SAMPLING_STEPS, "diffusion_schedule": DIFFUSION_SCHEDULE,
            "diffusion_prediction": DIFFUSION_PREDICTION, "diffusion_loss_weight": DIFFUSION_LOSS_WEIGHT,
            "training_stages": ["vae_pretrain", "diffusion_warmup"] + ([] if frozen_vae else ["joint"]),
            "frozen_vae": frozen_vae, "selection_metric": "内部验证真实标签 30-NN miLISI",
            "vae_state_before_diffusion": vae_before, "vae_state_after_training": vae_after,
            "latent_normalization": "预训练后由内部训练后验拟合并固定",
            "class_names": class_names.tolist(), "seed": SEED, "beta_target": beta_target,
            "epochs_completed": int(sum(r["stage"] != "diffusion_warmup" for r in history) + diffusion_epochs_completed),
            "best_epoch": best_epoch, "best_validation_loss": selection["selection_reconstruction"],
            **selection, **checkpoint_metadata,
        }
        partial = model_path.with_suffix(".pt.partial")
        torch.save(checkpoint, partial)
        partial.replace(model_path)
        result = {k: v for k, v in checkpoint.items() if k not in {"state_dict", "class_names"}}
        result.update(candidate_dir=str(candidate_dir), model_path=str(model_path), history_path=str(history_path))
        write_json(candidate_dir / "diagnostics.json", result)
        del model, best_state, checkpoint
        if device.type == "cuda":
            torch.cuda.empty_cache()
        return result

    return train_cvae_candidate(matrix, retained_rows, medians, labels, class_names,
                                train_positions, validation_positions, beta_target,
                                candidate_dir, device, checkpoint_metadata)


## 训练掩码

仅从内部训练fit行抽取同类型掩码，不使用验证或留出细胞；复制区域保留模型输出值。

In [ ]:
def sample_mask_rows(training_labels, fit_rows, candidate_labels, seed):
    rng = np.random.default_rng(seed)
    donors = np.empty(len(candidate_labels), dtype=int)
    for level in np.unique(candidate_labels):
        pool = fit_rows[training_labels[fit_rows] == level]
        if not len(pool):
            raise ValueError(f'训练掩码没有细胞类型：{level}')
        rows = np.flatnonzero(candidate_labels == level)
        donors[rows] = rng.choice(pool, len(rows), replace=True)
    return donors

def apply_training_mask(candidate, donor_values):
    if candidate.shape != donor_values.shape:
        raise ValueError('生成矩阵和训练掩码形状不一致')
    return np.where(np.isfinite(donor_values), candidate, np.nan).astype(np.float32)


def write_training_mask_h5ad(raw_path, output_path, matrix, training_labels, fit_rows,
                             seed=42, feature_block=4096):
    """同类型fit细胞整行缺失掩码；所有有限生成值和特征轴保持原样。"""
    raw_path, output_path = Path(raw_path), Path(output_path)
    if output_path.exists():
        raise FileExistsError(f'输出已存在：{output_path}')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    source = ad.read_h5ad(raw_path, backed='r')
    partial = output_path.with_name(output_path.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        if matrix.shape[1] != source.n_vars or len(training_labels) != matrix.shape[0]:
            raise ValueError('训练掩码与生成矩阵的区域轴或行标签不一致')
        labels = source.obs['Subclass'].astype(str).to_numpy()
        donors = sample_mask_rows(np.asarray(training_labels), np.asarray(fit_rows), labels, seed)
        obs = source.obs.copy()
        obs['mask_donor_row'] = donors
        obs['generator'] = 'CVAE＋训练掩码'
        shell = ad.AnnData(sp.csr_matrix(source.shape, dtype=np.float32), obs=obs, var=source.var.copy())
        shell.uns['postprocessing'] = {'method': 'training_mask', 'mask_seed': seed,
            'mask_reference': 'same Subclass, internal fit cells only', 'model_unchanged': True}
        shell.write_h5ad(partial, compression='gzip')
        observed = 0
        with h5py.File(partial, 'r+') as handle:
            del handle['X']
            output = handle.create_dataset('X', shape=source.shape, dtype='float32',
                chunks=(min(8, source.n_obs), min(feature_block, source.n_vars)),
                compression='gzip', compression_opts=1, shuffle=True, fillvalue=np.nan)
            output.attrs['encoding-type'], output.attrs['encoding-version'] = 'array', '0.2.0'
            for start in range(0, source.n_vars, feature_block):
                stop = min(start + feature_block, source.n_vars)
                candidate = source.X[:, start:stop]
                candidate = np.asarray(candidate.toarray() if hasattr(candidate, 'toarray') else candidate, dtype=np.float32)
                donor = np.asarray(matrix[np.ix_(donors, np.arange(start, stop))], dtype=np.float32)
                values = apply_training_mask(candidate, donor)
                np.testing.assert_array_equal(np.isfinite(values), np.isfinite(donor))
                np.testing.assert_array_equal(values[np.isfinite(values)], candidate[np.isfinite(values)])
                observed += int(np.isfinite(values).sum())
                output[:, start:stop] = values
        partial.replace(output_path)
        summary = {'方法': 'CVAE_training_mask', '数据': str(output_path), '掩码种子': seed,
            '细胞数': source.n_obs, '区域数': source.n_vars, '观测率': observed / np.prod(source.shape),
            '仅使用同类型内部训练掩码': True, '有限生成值未改变': True}
        (output_path.parent/'mask_summary.json').write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        return summary
    finally:
        source.file.close()
        if partial.exists():
            partial.unlink()


## 联合秩投影和显式分位数映射

同类型真实fit矩阵转为正态秩分数，PCA学习联合因子；生成秩分数投影后进行协方差白化／着色，再逐区域匹配真实经验分位数。缺失位置不变；fit少于5个或内部验证少于3个的类型保留原值。映射按整个同类型批次计算，不能逐细胞独立应用。高复制Pearson是本后处理约束的结果，不能等同于CVAE原始相关学习能力。

In [ ]:
def gaussian_ranks(values):
    """有限值转平均秩正态分数，缺失中心填0；各非恒定列方差标准化。"""
    finite=np.isfinite(values)
    counts=finite.sum(0)
    ranks=rankdata(np.where(finite,values,np.nan),axis=0,method='average',nan_policy='omit')
    quantiles=(ranks-0.5)/np.maximum(counts,1)
    scores=ndtri(np.clip(quantiles,1e-6,1-1e-6))
    return standardized_scores(scores,finite)

def standardized_scores(values,observed):
    counts=observed.sum(0)
    mean=np.divide(np.sum(values,where=observed,axis=0,dtype=np.float64),counts,
                   out=np.zeros(values.shape[1]),where=counts>0)
    centered=np.where(observed,values-mean,0)
    deviation=np.sqrt(np.sum(centered**2,axis=0)/max(len(values)-1,1))
    result=np.divide(centered,deviation,out=np.zeros_like(centered),where=deviation>1e-7)
    return result.astype(np.float32)

def symmetric_power(matrix,power):
    values,vectors=np.linalg.eigh((matrix+matrix.T)/2)
    floor=max(float(values.max())*1e-5,1e-8)
    return (vectors*np.maximum(values,floor)**power)@vectors.T

def fit_rank_factors(reference,n_components):
    scores=gaussian_ranks(reference)
    pca=PCA(n_components=n_components,svd_solver='randomized',iterated_power=3,random_state=42)
    factors=pca.fit_transform(scores)
    gram=factors.T.astype(np.float64)@factors.astype(np.float64)
    centered=scores-pca.mean_
    # 所有区域都走同一回归；不按复制配对表构造或重写任何列。
    basis=np.linalg.solve(gram+np.eye(n_components)*max(np.trace(gram)/n_components,1)*1e-7,
                          factors.T@centered).T.astype(np.float32)
    covariance=(gram/max(len(reference)-1,1)).astype(np.float64)
    return {'basis':basis,'covariance':covariance,'mean':pca.mean_.astype(np.float32),
            'explained_variance':float(pca.explained_variance_ratio_.sum())}

def shared_rank_projection(source_scores,observed,model,n_components):
    q=min(n_components,model['basis'].shape[1],len(source_scores)-1)
    basis=model['basis'][:,:q]
    coordinates=(source_scores-model['mean'])@basis
    coordinates=coordinates@np.linalg.pinv(basis.T@basis,rcond=1e-5)
    coordinates-=coordinates.mean(0)
    covariance=coordinates.T.astype(np.float64)@coordinates.astype(np.float64)/max(len(coordinates)-1,1)
    transform=symmetric_power(covariance,-0.5)@symmetric_power(model['covariance'][:q,:q],0.5)
    transported=(coordinates@transform).astype(np.float32)
    shared=transported@basis.T+model['mean']
    return standardized_scores(shared,observed)

def quantile_from_sorted(reference_sorted,reference_counts,scores,original):
    """逐列显式经验分位数匹配；模型分数只决定细胞间排序。"""
    observed=np.isfinite(original)
    counts=observed.sum(0)
    eligible=(reference_counts>0)&(counts>0)
    result=original.copy()
    if not eligible.any():
        return result
    ranks=rankdata(np.where(observed[:,eligible],scores[:,eligible],np.nan),axis=0,
                   method='average',nan_policy='omit')
    quantiles=(ranks-0.5)/counts[eligible]
    positions=np.nan_to_num(quantiles,nan=0.5)*reference_counts[eligible]-0.5
    positions=np.clip(positions,0,reference_counts[eligible]-1)
    lower=np.floor(positions).astype(int)
    upper=np.minimum(lower+1,reference_counts[eligible]-1)
    ordered=reference_sorted[:,eligible]
    left=np.take_along_axis(ordered,lower,axis=0)
    right=np.take_along_axis(ordered,upper,axis=0)
    mapped=left+(right-left)*(positions-lower)
    result[:,eligible]=np.where(observed[:,eligible],mapped,np.nan)
    return result

def map_class(original,reference_sorted,reference_counts,model,n_components,eta):
    if len(original)<3:
        return original.copy()
    source=gaussian_ranks(original)
    observed=np.isfinite(original)
    if eta:
        shared=shared_rank_projection(source,observed,model,n_components)
    output=np.empty_like(original,dtype=np.float32)
    for start in range(0,original.shape[1],4096):
        stop=min(start+4096,original.shape[1])
        score=source[:,start:stop] if not eta else (1-eta)*source[:,start:stop]+eta*shared[:,start:stop]
        output[:,start:stop]=quantile_from_sorted(reference_sorted[:,start:stop],reference_counts[start:stop],
            score,original[:,start:stop])
    assert np.array_equal(np.isnan(output),np.isnan(original))
    return output



def write_json(path,payload):
    """完成标记和运行签名在写完后原子替换。"""
    path=Path(path)
    temporary=path.with_name(path.name+'.partial')
    temporary.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
    temporary.replace(path)


def feature_axis_hash(names):
    return hashlib.sha256(json.dumps(list(map(str,names)),ensure_ascii=False).encode()).hexdigest()


def copy_checkpoint(source,destination):
    destination=Path(destination)
    temporary=destination.with_name(destination.name+'.partial')
    shutil.copy2(source,temporary)
    temporary.replace(destination)


def fit_joint_mapping(matrix,labels,fit_rows,validation_rows,feature_names,output_dir,components=32):
    """只在fit真实值上拟合；内部验证只提供类型样本数，决定小样本回退和维数。"""
    output_dir=Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    labels=np.asarray(labels,dtype=str)
    fit_rows,validation_rows=np.asarray(fit_rows,dtype=int),np.asarray(validation_rows,dtype=int)
    if np.intersect1d(fit_rows,validation_rows).size:
        raise ValueError('fit和内部验证细胞不能重叠')
    if len(feature_names)!=matrix.shape[1] or len(labels)!=matrix.shape[0]:
        raise ValueError('训练矩阵、标签或区域轴不一致')
    identity={'区域轴SHA256':feature_axis_hash(feature_names),'矩阵形状':list(matrix.shape),
              'fit行':fit_rows.tolist(),'验证行':validation_rows.tolist(),'标签':labels.tolist(),
              '因子上限':int(components),'投影强度':1.0}
    identity_path=output_dir/'identity.json'
    if identity_path.exists() and json.loads(identity_path.read_text(encoding='utf-8'))!=identity:
        raise ValueError('映射运行签名变化，请使用新输出目录')
    if not identity_path.exists():
        write_json(identity_path,identity)
    records=[]
    for index,level in enumerate(np.unique(labels[validation_rows])):
        rows=fit_rows[labels[fit_rows]==level]
        count=int(np.sum(labels[validation_rows]==level))
        record={'细胞类型':str(level),'fit细胞数':len(rows),'内部验证细胞数':count,
                '文件前缀':f'model_{index:03d}'}
        if len(rows)<5 or count<3:
            record['状态']='小样本保留原生成值'
            records.append(record)
            continue
        prefix=output_dir/record['文件前缀']
        model_path=prefix.with_suffix('.npz')
        sorted_path=prefix.with_name(prefix.name+'_sorted.npy')
        if not (model_path.exists() and sorted_path.exists()):
            reference=np.array(matrix[rows],dtype=np.float32)
            counts=np.isfinite(reference).sum(0).astype(np.int32)
            q=min(components,len(rows)-1,count-1)
            with threadpool_limits(limits=8):
                model=fit_rank_factors(reference,q)
            temporary_sorted=sorted_path.with_name(sorted_path.name+'.partial')
            temporary_model=model_path.with_name(model_path.name+'.partial')
            with temporary_sorted.open('wb') as stream:
                np.save(stream,np.sort(np.where(np.isfinite(reference),reference,np.inf),axis=0))
            temporary_sorted.replace(sorted_path)
            with temporary_model.open('wb') as stream:
                np.savez(stream,**model,reference_counts=counts)
            temporary_model.replace(model_path)
            del reference,model
            gc.collect()
        with np.load(model_path) as model:
            record.update({'状态':'已拟合','因子数':int(model['basis'].shape[1]),
                           '秩空间解释方差':float(model['explained_variance'])})
        records.append(record)
        print('联合秩映射拟合：',record,flush=True)
    metadata={**identity,'方法':'simscDNAm','版本':'1.0','类型明细':records}
    write_json(output_dir/'metadata.json',metadata)
    return metadata


def apply_joint_mapping(masked_path,output_path,models_dir):
    """固定联合投影强度1并匹配fit经验分位数；所有列遵循同一规则。"""
    masked_path,output_path,models_dir=Path(masked_path),Path(output_path),Path(models_dir)
    if output_path.exists():
        raise FileExistsError(f'输出已存在：{output_path}')
    metadata_path=models_dir/'metadata.json'
    metadata=json.loads(metadata_path.read_text(encoding='utf-8'))
    original=ad.read_h5ad(masked_path)
    if feature_axis_hash(original.var_names)!=metadata['区域轴SHA256']:
        raise ValueError('生成数据的区域名称或顺序与映射不一致')
    values=original.X.toarray() if sp.issparse(original.X) else original.X.copy()
    labels=original.obs['Subclass'].astype(str).to_numpy()
    details=[]
    for record in metadata['类型明细']:
        rows=np.flatnonzero(labels==record['细胞类型'])
        if record['状态']!='已拟合' or len(rows)<3:
            continue
        prefix=models_dir/record['文件前缀']
        with np.load(prefix.with_suffix('.npz')) as saved:
            model={key:saved[key].copy() for key in ['basis','covariance','mean']}
            counts=saved['reference_counts'].copy()
        ordered=np.load(prefix.with_name(prefix.name+'_sorted.npy'),mmap_mode='r')
        with threadpool_limits(limits=8):
            values[rows]=map_class(values[rows],ordered,counts,model,metadata['因子上限'],1.)
        details.append({'细胞类型':record['细胞类型'],'映射细胞数':len(rows)})
        del model,ordered
        gc.collect()
        print('联合秩映射应用：',details[-1],flush=True)
    source_values=original.X.toarray() if sp.issparse(original.X) else original.X
    np.testing.assert_array_equal(np.isnan(values),np.isnan(source_values))
    if np.nanmin(values)<0 or np.nanmax(values)>1:
        raise ValueError('主模型生成值超出0–1')
    result=ad.AnnData(values,obs=original.obs.copy(),var=original.var.copy(),uns=dict(original.uns))
    result.obs['generator']='simscDNAm'
    result.uns['simscDNAm']={'方法':'simscDNAm','版本':'1.0',
        '流程':'CVAE + training mask + joint rank mapping','因子上限':metadata['因子上限'],
        '投影强度':1.,'fit细胞数':len(metadata['fit行']),
        '映射参数SHA256':hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
        '未使用复制配对重写输出':True}
    output_path.parent.mkdir(parents=True,exist_ok=True)
    partial=output_path.with_suffix('.h5ad.partial')
    result.write_h5ad(partial,compression='gzip',compression_opts=1)
    partial.replace(output_path)
    report={'方法':'simscDNAm','状态':'完成','总细胞数':len(labels),
            '映射细胞数':sum(item['映射细胞数'] for item in details),'类型明细':details,
            '数据':str(output_path),'区域数':result.n_vars,'NaN位置保持':True}
    write_json(output_path.parent/'alignment_summary.json',report)
    return report


def finalize_model(dataset_root,matrix,training_labels,fit_rows,validation_rows,names,masked_path,checkpoint_path):
    model_root=Path(dataset_root)/'simscDNAm'
    mapped_path=model_root/'data.h5ad'
    models_dir=model_root/'alignment'
    if not mapped_path.exists():
        fit_joint_mapping(matrix,training_labels,fit_rows,validation_rows,names,models_dir)
        apply_joint_mapping(masked_path,mapped_path,models_dir)
    model_root.mkdir(parents=True,exist_ok=True)
    if not (model_root/'model.pt').exists():
        copy_checkpoint(checkpoint_path,model_root/'model.pt')
    return mapped_path


## 输入与运行

默认演示Liu2023全量输入。更换数据时修改DATASETS、RUN_DATASETS和RUN_ROOT；输入须带真实Subclass和训练前追加的1000个复制区间。新训练使用四个内部候选；PRETRAINED_RUNS仅接受训练输入一致的既有CVAE＋mask目录。对齐参数固定为本次已确认版本，不在外层留出上重新选参。

In [ ]:
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
SERVER = sys.platform != "win32"
DATA_ROOT = Path("/home/sccasimp/data/xb/simscDNAm") if SERVER else PROJECT / "data/raw"
OUTPUTS = Path("/home/sccasimp/data/xb/outputs/simscDNAm") if SERVER else PROJECT / "outputs"
DATASET = "liu2023/full"
RUN_ROOT = OUTPUTS / "model"
INPUT_ROOT = OUTPUTS / "preprocessing" / DATASET
DATASETS = {DATASET: {
    "training_h5ad": INPUT_ROOT / "TrainingReal.h5ad",
    "target_h5ad": INPUT_ROOT / "HeldoutReal.h5ad",
    "region_pairs_csv": INPUT_ROOT / "region_pairs.csv"}}
RUN_DATASETS = (DATASET,)
# 留空从头训练；复用时填写 {DATASET: Path("已有训练目录")}。
PRETRAINED_RUNS = {}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG = TrainingConfig(BETA_TARGETS=(1e-4, 1e-5, 1e-6))
CASES = (("joint_beta_1e4", 1e-4, False), ("frozen_beta_1e4", 1e-4, True),
         ("frozen_beta_1e5", 1e-5, True), ("frozen_beta_1e6", 1e-6, True))
assert Path(sys.prefix).name.lower() == "simscdnam", sys.executable
torch.set_num_threads(8)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("运行配置：", {"数据集": RUN_DATASETS, "设备": str(DEVICE), "方法": "simscDNAm", "候选": CASES})

In [ ]:
def write_status(dataset_root, phase, **extra):
    payload = {"阶段": phase, "更新时间": datetime.now(timezone(timedelta(hours=8))).isoformat(), **extra}
    partial = dataset_root / "status.json.partial"
    partial.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    partial.replace(dataset_root / "status.json")


def expanded_heldout(raw_path,output_path,training,pairs):
    """原始轴追加真实复制列；完整轴核验后保留，合成列不在此处理。"""
    output_path=Path(output_path)
    if output_path.exists():
        raise FileExistsError(f'真实留出输出已存在：{output_path}')
    raw=ad.read_h5ad(raw_path,backed='r')
    try:
        expanded=raw.var_names.equals(training.var_names)
        if not expanded and not raw.var_names.equals(training.var_names[:-1000]):
            raise ValueError('留出与训练的原始或扩维区域轴不一致')
        if len(raw.obs_names.intersection(training.obs_names)):
            raise ValueError('真实训练与外层留出细胞重叠')
        if expanded:
            validate_real_duplicates(raw,pairs)
        output_path.parent.mkdir(parents=True,exist_ok=True)
        shell=ad.AnnData(X=sp.csr_matrix((raw.n_obs,training.n_vars),dtype=np.float32),
                         obs=raw.obs.copy(),var=training.var.copy())
        partial=output_path.with_suffix('.h5ad.partial')
        shell.write_h5ad(partial)
        original=pairs['原region索引'].to_numpy(dtype=int)
        with h5py.File(partial,'r+') as handle:
            del handle['X']
            values=handle.create_dataset('X',shape=(raw.n_obs,training.n_vars),dtype='float32',
                chunks=(min(16,raw.n_obs),min(4096,training.n_vars)),compression='gzip',compression_opts=4)
            values.attrs['encoding-type'],values.attrs['encoding-version']='array','0.2.0'
            for start in range(0,raw.n_obs,16):
                block=raw.X[start:start+16]
                block=np.asarray(block.toarray() if sp.issparse(block) else block,dtype=np.float32)
                values[start:start+len(block)]=block if expanded else np.column_stack([block,block[:,original]])
        partial.replace(output_path)
    finally:
        raw.file.close()


def run_dataset(dataset):
    training_path = Path(DATASETS[dataset]["training_h5ad"])
    target_path = Path(DATASETS[dataset]["target_h5ad"])
    pair_path = Path(DATASETS[dataset]["region_pairs_csv"])
    dataset_root = RUN_ROOT / dataset
    dataset_root.mkdir(parents=True, exist_ok=True)
    if dataset in PRETRAINED_RUNS:
        return run_pretrained_dataset(dataset, PRETRAINED_RUNS[dataset], dataset_root)
    notebook = json.loads((PROJECT / "model/script/train.ipynb").read_text(encoding="utf-8"))
    definitions = "\n".join("".join(c["source"]) for c in notebook["cells"] if c.get("metadata", {}).get("definitions"))
    inputs = [{"path": str(p), "size": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns}
              for p in (training_path, target_path, pair_path)]
    guard_run_identity(dataset_root, {"config": CONFIG.to_dict(), "cases": CASES, "inputs": inputs,
        "notebook_definitions_sha256": hashlib.sha256(definitions.encode()).hexdigest(),
        "dependencies": {name: hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() for name in
            ("model/src/model.py", "model/src/config.py", "data/training.py", "model/script/inference.ipynb",
             "evaluate/A3/metrics.py", "data/evaluation.py", "evaluate/A3/evaluate.ipynb")}})
    if (dataset_root / "completed.json").exists():
        print("复用签名一致的已完成数据集：", dataset, flush=True)
        return json.loads((dataset_root / "completed.json").read_text(encoding="utf-8"))
    write_status(dataset_root, "读取训练数据")
    data = load_training_input(training_path, dataset_root / "work", pair_path, CONFIG.INPUT_ROW_BLOCK)
    matrix = data["matrix"]
    try:
        retained, metadata = data["retained_rows"], data["cell_metadata"]
        labels_text = metadata.loc[retained, "Subclass"].to_numpy(dtype=str)
        classes = np.sort(np.unique(labels_text))
        labels = np.searchsorted(classes, labels_text)
        fit, validation = internal_subclass_split(np.arange(len(retained)), labels_text,
            CONFIG.INTERNAL_VALIDATION_FRACTION, CONFIG.SEED)
        medians = fit_feature_medians(matrix, retained[fit], CONFIG.FEATURE_BLOCK)
        labels_by_row = np.full(matrix.shape[0], -1, dtype=int)
        labels_by_row[retained] = labels
        reference = fit_selection_reference(matrix, retained[fit], retained[validation], labels_by_row,
            natural_count=matrix.shape[1]-1000)
        np.savez_compressed(dataset_root / "internal_reference.npz", selected=reference["selected"],
            fill=reference["fill"], pca_mean=reference["pca"].mean_, pca_components=reference["pca"].components_,
            fit_rows=retained[fit], validation_rows=retained[validation])
        names, var = data["feature_names"], data["feature_var"]
        data["region_pairs"].to_csv(dataset_root / "region_pairs.csv", index=False)
        (dataset_root / "config.json").write_text(json.dumps({"配置": CONFIG.to_dict(),
            "候选": CASES, "选模": "只用内部验证，真实留出矩阵不参与", "标签": "Subclass",
            "输入": str(training_path), "输入区域数": int(matrix.shape[1]),
            "内部训练细胞": int(len(fit)), "内部验证细胞": int(len(validation)),
            "评分区域数": int(len(reference["selected"])), "评分PC数": int(reference["pca"].n_components_)},
            ensure_ascii=False, indent=2), encoding="utf-8")
        records = []
        for name, beta, frozen in CASES:
            candidate = dataset_root / name
            if (candidate / "diagnostics.json").exists():
                record = json.loads((candidate / "diagnostics.json").read_text(encoding="utf-8"))
            else:
                write_status(dataset_root, "训练", 候选=name)
                record = train_candidate(matrix, retained, medians, labels, classes, fit, validation,
                    beta, candidate, DEVICE, {"natural_features": matrix.shape[1]-1000,
                    "duplicate_features": 1000}, CONFIG, selection_reference=reference, frozen_vae=frozen)
            record["case"] = name
            records.append(record)
            matched = [r for r in records if float(r["beta_target"]) == 1e-4]
            if len(matched) == 2 and len({r["vae_state_before_diffusion"] for r in matched}) != 1:
                raise AssertionError("联合和冻结对照没有使用相同的预训练初始化")
            pd.DataFrame(records).to_csv(dataset_root / "internal_selection.csv", index=False)
            gc.collect()
        eligible = [record for record in records if record["selection_eligible"]]
        if not eligible:
            raise RuntimeError("所有候选均未通过原重构/多样性验收；不提升正式模型")
        selected = min(eligible, key=selection_rank)
        write_json(dataset_root / "selected.json", selected)
        # 外层留出仅在选模完成后读取；固定每种生成方法的类别配额。
        target_labels = load_target_labels(target_path, names, classes)
        np.random.default_rng(CONFIG.SEED).shuffle(target_labels)
        target_classes = np.searchsorted(classes, target_labels)
        final_names = [selected["case"]]
        for name in final_names:
            record = next(record for record in records if record["case"] == name)
            model, checkpoint = load_cvae_model(Path(record["model_path"]), DEVICE)
            destination = dataset_root / name / "data.h5ad"
            if not destination.exists():
                write_synthetic_h5ad(destination, model, target_classes, target_labels, var, names,
                    checkpoint["sampling_steps"], DEVICE, CONFIG.SEED,
                    CONFIG.GENERATION_ROW_BLOCK, CONFIG.FEATURE_BLOCK)
            validate_synthetic_h5ad(destination, names, len(target_labels))
            del model
            torch.cuda.empty_cache()
        raw_path = dataset_root / selected["case"] / "data.h5ad"
        masked_path = dataset_root / "CVAE_training_mask/data.h5ad"
        if not masked_path.exists():
            write_training_mask_h5ad(raw_path, masked_path, matrix,
                metadata['Subclass'].to_numpy(dtype=str), retained[fit], CONFIG.SEED)
        checkpoint_path = masked_path.parent / 'model.pt'
        if not checkpoint_path.exists():
            copy_checkpoint(selected['model_path'], checkpoint_path)
        raw_path.unlink()
        mapped_path = finalize_model(dataset_root, matrix, metadata['Subclass'].to_numpy(dtype=str),
            retained[fit], retained[validation], names, masked_path, checkpoint_path)
        training = ad.read_h5ad(training_path, backed="r")
        try:
            heldout_path = dataset_root / "HeldoutReal/data.h5ad"
            if not heldout_path.exists():
                expanded_heldout(target_path, heldout_path, training, data["region_pairs"])
        finally:
            training.file.close()
        report = {"状态": "完成", "入选候选": selected["case"], "选模依据": "内部验证 miLISI",
                  "方法": "simscDNAm", "模型": str(mapped_path.parent / "model.pt"),
                  "数据": str(mapped_path), "训练输入": str(training_path), "真实留出": str(heldout_path),
                  "因子上限": 32, "投影强度": 1.0}
        write_json(dataset_root / "completed.json", report)
        write_status(dataset_root, "完成", **report)
        return report
    finally:
        matrix._mmap.close()


def validate_pretrained_inputs(training_path,cached_path,audit_path,masked_path,target_path):
    """逐值核对既有训练缓存，同时确认当前生成配额仍适用。"""
    truth=ad.read_h5ad(training_path,backed='r')
    masked=ad.read_h5ad(masked_path,backed='r')
    cached=np.load(cached_path,mmap_mode='r')
    try:
        audit=pd.read_csv(audit_path)
        labels=truth.obs['Subclass'].astype(str).to_numpy()
        if (truth.shape!=cached.shape or not truth.var_names.equals(masked.var_names)
            or not np.array_equal(truth.obs_names.astype(str),audit['cell_id'].astype(str))
            or not np.array_equal(labels,audit['Subclass'].astype(str))):
            raise ValueError('训练缓存的细胞、标签或区域轴与当前训练输入不一致')
        for start in range(0,truth.n_obs,64):
            block=truth.X[start:start+64]
            values=np.asarray(block.toarray() if sp.issparse(block) else block,dtype=np.float32)
            if not np.array_equal(values,cached[start:start+64],equal_nan=True):
                raise ValueError('当前训练矩阵与checkpoint原训练缓存不一致')
        target=load_target_labels(target_path,truth.var_names.to_numpy(),np.unique(labels))
        counts=pd.Series(target).value_counts().sort_index()
        generated=masked.obs['Subclass'].astype(str).value_counts().sort_index()
        if not (counts.index.equals(generated.index) and np.array_equal(counts.to_numpy(),generated.to_numpy())):
            raise ValueError('当前目标类型配额与已有生成结果不一致，请从头生成')
        return labels,truth.var_names.to_numpy()
    finally:
        truth.file.close()
        masked.file.close()
        cached._mmap.close()


def run_pretrained_dataset(dataset,source,dataset_root):
    """复用已完成的CVAE＋mask产物，重新拟合正式对齐并另存最终模型。"""
    source=Path(source)
    training_path=Path(DATASETS[dataset]['training_h5ad'])
    target_path=Path(DATASETS[dataset]['target_h5ad'])
    configuration=json.loads((source/'config.json').read_text(encoding='utf-8'))
    if Path(configuration['输入']).resolve()!=training_path.resolve():
        raise ValueError('checkpoint来源与配置的训练输入不一致')
    masked_path=source/'CVAE_training_mask/data.h5ad'
    checkpoint=source/'CVAE_training_mask/model.pt'
    paths=[training_path,target_path,masked_path,checkpoint,source/'internal_reference.npz',
           source/'work/values.float32.npy',source/'work/input_cell_audit.csv']
    guard_run_identity(dataset_root,{'方法':'simscDNAm','模式':'reuse','源目录':str(source),
        '输入':[{'path':str(p),'size':p.stat().st_size,'mtime_ns':p.stat().st_mtime_ns} for p in paths],
        '因子上限':32,'投影强度':1.,'源码SHA256':release_source_hashes()})
    if (dataset_root/'completed.json').exists():
        return json.loads((dataset_root/'completed.json').read_text(encoding='utf-8'))
    with np.load(source/'internal_reference.npz') as reference:
        fit,validation=reference['fit_rows'].copy(),reference['validation_rows'].copy()
    labels,names=validate_pretrained_inputs(training_path,source/'work/values.float32.npy',
        source/'work/input_cell_audit.csv',masked_path,target_path)
    matrix=np.load(source/'work/values.float32.npy',mmap_mode='r')
    try:
        mapped=finalize_model(dataset_root,matrix,labels,fit,validation,names,masked_path,checkpoint)
    finally:
        matrix._mmap.close()
    shutil.copy2(source/'region_pairs.csv',dataset_root/'region_pairs.csv')
    heldout_path=dataset_root/'HeldoutReal/data.h5ad'
    truth=ad.read_h5ad(training_path,backed='r')
    try:
        if not heldout_path.exists():
            expanded_heldout(target_path,heldout_path,truth,pd.read_csv(dataset_root/'region_pairs.csv'))
    finally:
        truth.file.close()
    report={'状态':'完成','方法':'simscDNAm','模式':'reuse','数据':str(mapped),
            '模型':str(mapped.parent/'model.pt'),'训练输入':str(training_path),
            '训练缓存目录':str(source.resolve()),
            '真实留出':str(heldout_path),'源checkpoint':str(checkpoint),
            '因子上限':32,'投影强度':1.,'内部fit细胞数':len(fit),'内部验证细胞数':len(validation)}
    write_json(dataset_root/'completed.json',report)
    write_status(dataset_root,'完成',**report)
    return report


def release_source_hashes():
    names=['model/script/train.ipynb','model/src/model.py','model/src/config.py',
           'data/training.py','model/script/inference.ipynb']
    result={name:hashlib.sha256((PROJECT/name).read_bytes()).hexdigest() for name in names[1:]}
    notebook=json.loads((PROJECT/names[0]).read_text(encoding='utf-8'))
    code='\n'.join(''.join(c['source']) for c in notebook['cells']
                   if c['cell_type']=='code' and c.get('metadata',{}).get('definitions'))
    result[names[0]]=hashlib.sha256(code.encode()).hexdigest()
    return result


In [ ]:
reports = {}
for dataset in RUN_DATASETS:
    print("开始数据集：", dataset, flush=True)
    reports[dataset] = run_dataset(dataset)
comparison = pd.DataFrame([{'数据集': dataset, **report} for dataset, report in reports.items()])
comparison.to_csv(RUN_ROOT/'generation_summary.csv', index=False)
display(comparison)
